# Cropland Masking

By default, evy restricts EVI statistics to cropland pixels (`mask_cropland=True`). This recipe compares masked vs. unmasked results to show the effect of the cropland filter on the EVI signal.

**Important for paper methodology:** The GEE backend uses Dynamic World (class 4) for its cropland mask, while the local backend uses ESA WorldCover (class 40). These products may disagree at the pixel level. When writing up your methodology, state which backend you used. See [Design Decisions](../../docs/design-decisions.md) for details.

> **Recipe pattern:** This notebook follows a two-stage layout — **Fetch (cached)** and **Analyze (fast)**. The fetch step uses [`evy.cached_zonal_stats`](../../docs/api-reference.md) which stores results on disk; re-running the analysis or tweaking a chart no longer triggers a refetch. See the cache section in `docs/troubleshooting.md` for invalidation.


In [ ]:
# papermill parameters
country = "SMR"
admin_level = 1
start_date = "2023-01-01"
end_date = "2023-12-31"
quick_mode = False


In [ ]:
if quick_mode:
    end_date = "2023-03-31"


In [ ]:
import evy

In [ ]:
gdf = evy.get_boundaries(country, admin_level=admin_level)
gdf[["shapeName"]].head()

## Fetch (cached): with cropland masking (default)

This is the default behavior — only cropland pixels contribute to the zonal statistics. The call uses `cached_zonal_stats`, so a second run with the same parameters hits the parquet cache in milliseconds.


In [ ]:
df_masked = evy.cached_zonal_stats(
    gdf,
    zone_col="shapeName",
    start_date=start_date,
    end_date=end_date,
    freq=evy.MONTHLY,
    stats=["mean"],
    mask_cropland=True,  # default, shown explicitly
)
df_masked.head()


## Fetch (cached): without cropland masking

All land-cover types contribute — forest, grassland, urban, water, etc. The two fetch calls use different `mask_cropland` values, so they hash to different cache files and both are kept on disk for fast re-analysis.


In [ ]:
df_unmasked = evy.cached_zonal_stats(
    gdf,
    zone_col="shapeName",
    start_date=start_date,
    end_date=end_date,
    freq=evy.MONTHLY,
    stats=["mean"],
    mask_cropland=False,
)
df_unmasked.head()


## Analyze (fast): side-by-side comparison

Plot both series to see how the mask affects the seasonal signal. Editing chart styling or filtering down to a single region only re-runs these cells — the fetch parquet on disk stays warm.


In [ ]:
df_masked

In [ ]:
evy.plot_time_series(df_masked, value_col="mean", title="EVI — Cropland Only")

In [ ]:
evy.plot_time_series(df_unmasked, value_col="mean", title="EVI — All Land Cover")

In regions with mixed land cover, the unmasked signal typically shows lower amplitude because non-agricultural vegetation (forest, grassland) has different seasonal dynamics than crops. The difference is most pronounced in regions where cropland is a small fraction of total land area.

If you need a **custom** land-cover mask beyond cropland, you can use the low-level pipeline (`load_modis` → `load_landcover` → `compute_zonal_stats`) to inspect and modify the raster directly. See the [quickstart](../quickstart.ipynb) for an example of the low-level pipeline.